# FT-4 eval 시각화 (latent_sde 체크포인트)

체크포인트로 FT-4 네 task를 eval하면서 tick마다 로그와 영상을 남기고, 그것을 시각화합니다. 코드는 `ft4_viz.py`(같은 폴더, `lerobot.policies.latent_sde.ft4_viz`)에 있습니다.
- **프로토콜:** `trial.sh`, wandb eval과 같습니다. contact rules를 쓰고, 컨트롤러는 체크포인트에 맞춥니다(`impedance_targets`면 Tier-3 impedance, 아니면 OSC).
- **카메라 오차(`CAMERA_ERROR`):** 기본값 1.0은 `trial.sh`, wandb eval과 같습니다. 에피소드마다 agentview는 ±5 mm, ±0.5°, wrist는 ±2 mm, ±0.5° 안에서 섭동을 새로 뽑습니다. 0이면 demo를 녹화할 때처럼 카메라 위치가 정확합니다. 섭동은 env seed로만 정해집니다.
- **z 샘플:** env를 하나씩 돌리고 에피소드마다 torch seed를 다시 맞춥니다. 그래서 env seed가 같아도 z 샘플은 `trial.sh`의 batch eval(10 env 동시)과 다릅니다.
- **캐시:** 결과는 `OUT/<tag>/<task>_s<seed>_cam<CAMERA_ERROR>.pkl`과 `.mp4`에 저장됩니다. 이미 있으면 다시 돌리지 않고, 체크포인트 경로가 바뀐 경우에만 다시 돌립니다. 카메라 오차마다 캐시가 따로 있어서 값을 바꿔 가며 다시 실행해도 됩니다.
- **실행 시간:** 에피소드당 대략 10–40초입니다(Wipe가 가장 깁니다).
- **커널:** `Python (latent_sde)`.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.update(HF_HOME="/PublicSSD/himchan/hf_cache", MUJOCO_GL="egl", OMP_NUM_THREADS="4", MKL_NUM_THREADS="4")
os.chdir("/PublicSSD/himchan/lerobot")

R = "outputs/train/latent_sde_ft4v2_reactive_ft_tcp_onestep_constant_{}_30k/checkpoints/last/pretrained_model"
CKPTS = {  # tag: pretrained_model dir; several tags run one after another
    "wnsn": R.format("z16_wn03_g03_sn1_g03"),  # wrench + state noise
    # "wn": R.format("z16_wn03_g03"),           # wrench noise
    # "base": R.format("z16"),                  # no noise
}
TASKS = ["square_ft", "stack_ft", "wipe_ft", "door_ft"]
SEEDS = [1000, 1001, 1002]  # env seeds (trial.sh: 1000-1019)
CAMERA_ERROR = 1.0  # camera calibration error scale (trial.sh, wandb eval: 1.0; 0: nominal cameras as in the demos)
OUT = "/PublicSSD/himchan/ft4/impedance/viz"

In [ ]:
%matplotlib inline
from lerobot.policies.latent_sde import ft4_viz as V

recs = V.run(CKPTS, TASKS, SEEDS, OUT, CAMERA_ERROR)  # {tag: {task: [record per seed]}}

## 1. 요약
에피소드별 결과입니다. `end`는 success, timeout, 또는 실패 원인(contact rule 위반 등)입니다.

In [ ]:
TAG = next(iter(CKPTS))
V.summary(recs)

## 2. 에피소드 보기
- **영상:** 왼쪽은 agentview, 오른쪽은 wrist 카메라입니다. 위쪽 글줄에는 tick 번호와 그 tick 직전의 task 상태가 나옵니다. `z` 표시는 그 tick에서 z를 다시 샘플했다는 뜻입니다.
- **그래프:** 회색 세로선은 z 재샘플 tick입니다. 정책의 목표(impedance는 x̂(t+1)·W̄(t+1), OSC는 set point)는 t+1 위치에 그립니다.

In [ ]:
TASK, EP = "wipe_ft", 1
r = recs[TAG][TASK][EP]
V.video(r)

In [ ]:
V.plot_episode(r)

## 3. Wipe 진단
마커 하나는 다음 세 조건이 **같은 tick에** 모두 맞을 때만 지워집니다.
1. robosuite 판정: 마커가 패드 네 모서리 안에 있고, 패드 바닥면보다 2 cm 이내로 위에 있음
2. 패드–테이블 힘이 5–25 N
3. 압력 중심(CoP)이 패드 가운데 60 % 안에 있음 (긴 축 ±3.6 cm, 짧은 축 ±1.5 cm)

40 N을 넘거나, 팔이 테이블에 닿거나, 관절 한계에 걸리면 그 즉시 실패입니다. 성공하려면 마커 100개를 모두 지워야 합니다.

각 tick의 패드 상태는 아래 범주로 나눕니다. 판정 순서는 free → low → high → uneven → ok입니다.

| 범주 | 뜻 | 색 |
|---|---|---|
| free | 패드–테이블 힘 없음 | 회색 |
| low | 5 N 미만 | 노랑 |
| high | 25 N 초과 | 빨강 |
| uneven | 힘은 맞지만 CoP가 가운데 60 % 밖 | 파랑 |
| ok | 덮은 마커가 지워짐 | 초록 |

`wipe_funnel`은 마커 100개를 다음처럼 나눕니다.
- 지워짐
- 덮였지만 규칙 때문에 못 지움(`blocked_*`, 덮였던 tick에서 가장 많았던 범주 기준)
- 한 번도 덮이지 않음(`never_covered`, 경로 문제)

접촉 tick은 범주별 비율과, 아직 안 지워진 마커 위에 있었던 비율(`on_marker_%`)로 보여줍니다.

In [ ]:
V.wipe_funnel([r for tag in recs for r in recs[tag].get("wipe_ft", [])])

In [ ]:
V.wipe_maps(recs[TAG]["wipe_ft"])

### 에피소드 하나 자세히
- **지도:** 마커 색은 지워진 tick이고, 속이 빈 마커는 끝까지 안 지워진 것입니다. 점은 tick마다의 패드 중심(범주 색), 사각형은 접촉 중의 패드 자리입니다. ▲는 시작점입니다.
- **시간축 그래프:** 다섯 줄입니다.
  1. 패드 힘과 5–25 N 대역, 정책의 W̄(t+1)
  2. CoP와 ±60 % 선
  3. 기울기: 측정값, 목표, 패드 축 성분
  4. 패드 높이와 지운 마커 수
  5. tick마다 패드 밑에 있던 미삭제 마커 수(막대 색 = 그 tick의 범주, 초록이면 지워진 것)

In [ ]:
EP = 0
r = recs[TAG]["wipe_ft"][EP]
V.wipe_map(r)

In [ ]:
V.wipe_timeline(r)

In [ ]:
V.frames_at(r, V.wipe_events(r))  # or a list of ticks